In [ ]:
# CELL 1 — Install dependencies
!pip install easyocr gspread google-auth openpyxl --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 19.1 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 2 — Upload your invoice images
# ============================================================
from google.colab import files
uploaded = files.upload()  # Select all 10 invoice images here

Saving X00016469620.jpg to X00016469620.jpg
Saving X51005255805.jpg to X51005255805.jpg
Saving X51005268200.jpg to X51005268200.jpg
Saving X51005268262.jpg to X51005268262.jpg
Saving X51005268400.jpg to X51005268400.jpg
Saving X51005268472.jpg to X51005268472.jpg
Saving X51005301659.jpg to X51005301659.jpg
Saving X51005301661.jpg to X51005301661.jpg
Saving X51005301667.jpg to X51005301667.jpg
Saving X51005303661.jpg to X51005303661.jpg


In [ ]:
# ============================================================
# CELL 4 — Authenticate Google Sheets
# ============================================================
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Create a new Google Sheet (or open existing one by name)
spreadsheet = gc.create("Invoice_Automation_Output")

# Make it publicly viewable — anyone with link can view
spreadsheet.share(None, perm_type='anyone', role='reader')

worksheet = spreadsheet.sheet1

# Print the public link
print("✅ Google Sheet created!")
print("🔗 Public Link:", spreadsheet.url)

✅ Google Sheet created!
🔗 Public Link: https://docs.google.com/spreadsheets/d/12YUZ7Zh-HvZENG7VL-IJpB58UVxccYjJWeZWeFVegSk


In [ ]:
# ============================================================
# CELL 5 — All helper functions (Fixed version)
# ============================================================
import easyocr
import pandas as pd
import re
from datetime import datetime

# Initialize EasyOCR reader
reader = easyocr.Reader(['en'], gpu=False)

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_amount(text):
    amounts = re.findall(r'\d+\.\d{2}', text)
    if amounts:
        return float(max(amounts, key=lambda x: float(x)))
    return 0.0

def extract_date(text):
    patterns = [
        r'(?:due|date|payment|expire)[^\d]*(\d{2}[\/\-]\d{2}[\/\-]\d{2,4})',
        r'(\d{2}[\/\-]\d{2}[\/\-]\d{4})',
        r'(\d{2}[\/\-]\d{2}[\/\-]\d{2})',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    return "Not Found"

def extract_vendor(ocr_results):
    skip_words = ["receipt", "invoice", "return", "please", "strictly",
                  "bill", "no.", "tax", "tel", "fax", "date", "time",
                  "thank", "welcome", "total", "cash", "change"]
    top_lines = ocr_results[:4]
    for line in top_lines:
        text = line.strip()
        if len(text) > 3 and not any(w in text.lower() for w in skip_words):
            return text.upper()
    return top_lines[0].upper() if top_lines else "UNKNOWN"

def assign_category(vendor):
    vendor = vendor.lower()
    if any(w in vendor for w in ["restaurant", "food", "cafe", "bistro", "kitchen"]):
        return "Food & Beverage"
    elif any(w in vendor for w in ["mart", "store", "shop", "market", "retail"]):
        return "Retail"
    elif any(w in vendor for w in ["light", "electric", "water", "utility", "power"]):
        return "Utilities"
    else:
        return "Others"

def calculate_ageing(date_str):
    if date_str == "Not Found":
        return "Not Found"
    try:
        date_str = date_str.replace("-", "/")
        for fmt in ["%d/%m/%Y", "%m/%d/%Y", "%d/%m/%y", "%m/%d/%y"]:
            try:
                due_date = datetime.strptime(date_str, fmt)
                return (datetime.today() - due_date).days
            except:
                continue
        return "Not Found"
    except:
        return "Not Found"

print("✅ All functions loaded!")

✅ All functions loaded!


In [ ]:
# ============================================================
# CELL 6 — Process all images with EasyOCR (Fixed version)
# ============================================================
data = []
as_of_date = datetime.today().strftime("%d-%m-%Y")

for file_name in uploaded.keys():
    try:
        print(f"⏳ Processing: {file_name}...")

        # EasyOCR reads image — returns list of text lines
        results = reader.readtext(file_name, detail=0)
        text = clean_text(" ".join(results))

        print(f"   📄 OCR Preview: {text[:100]}...")

        vendor   = extract_vendor(results)   # passes raw list
        amount   = extract_amount(text)
        due_date = extract_date(text)
        category = assign_category(vendor)
        ageing   = calculate_ageing(due_date)

        data.append([file_name, vendor, amount, due_date, category, ageing, as_of_date])
        print(f"   ✅ Done | Vendor: {vendor} | Due: {due_date} | Ageing: {ageing} days")

    except Exception as e:
        print(f"❌ Error: {file_name} → {e}")

print(f"\n🎉 Finished! Processed {len(data)} invoices.")

⏳ Processing: X00016469620.jpg...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   📄 OCR Preview: tan woon yann MR 0.I.Y JohuR) SDN BHD (CO,REC 933109-X ) LoT 185/-4 & 185 i-B_ JALAN KPB Kamasan PER...
   ✅ Done | Vendor: TAN WOON YANN | Due: 12-01-19 | Ageing: 2645 days
⏳ Processing: X51005255805.jpg...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   📄 OCR Preview: SAM SAM TRADING co (742016-W) 67,JLN MEWAH 25/63 TmN SRI MUDA, 40400 Shah ALAH. TEL/FAX 03-51213881 ...
   ✅ Done | Vendor: TRADING | Due: 29-12-2017 | Ageing: 3024 days
⏳ Processing: X51005268200.jpg...


...

   📄 OCR Preview: LIGHTROOM GALLERY SDN BHD No: 28, JALAN ASTANA 1C, BANDAR BUkIT RAjA, 41050 KLANG SELANGOR D. E, MAL...
   ✅ Done | Vendor: LIGHTROOM  GALLERY SDN | Due: 20/12/2017 | Ageing: 3033 days
⏳ Processing: X51005301667.jpg...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   📄 OCR Preview: LIGHTROOM GALLERY No: 28, JALAN ASTANA 1G BANDAR BUKIT RAJA, KLANG SELANGOR D. E ROC No. (1072825-4)...
   ✅ Done | Vendor: LIGHTROOM GALLERY | Due: 20/11/2017 | Ageing: 3063 days
⏳ Processing: X51005303661.jpg...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   📄 OCR Preview: LIGHTROOM GALLERY SDN BHD No: 28, JALAN ASTANA 1C, BANDAR BUkIT RAjA, 41050 KLANG SELANGOR D. E, MAL...
   ✅ Done | Vendor: LIGHTROOM  GALLERY SDN | Due: 20/12/2017 | Ageing: 3033 days

🎉 Finished! Processed 10 invoices.


In [ ]:
# ============================================================
# CELL 7 — Write to Google Sheets + Download Excel backup
# ============================================================

# Create DataFrame
df = pd.DataFrame(data, columns=[
    "File Name", "Vendor", "Invoice Amount",
    "Due Date", "Category", "Ageing (Days)", "As Of Date"
])

print(df)

# --- Write to Google Sheets ---
# Add header row
worksheet.update([df.columns.tolist()] + df.values.tolist())

print("\n✅ Data written to Google Sheets!")
print("🔗 Public Link:", spreadsheet.url)

# --- Also save Excel as backup ---
excel_path = "Final_Invoice_Output.xlsx"
df.to_excel(excel_path, index=False)
files.download(excel_path)
print("✅ Excel backup downloaded!")

          File Name                  Vendor  Invoice Amount    Due Date  \
0  X00016469620.jpg           TAN WOON YANN           33.90    12-01-19   
1  X51005255805.jpg                 TRADING           13.30  29-12-2017   
2  X51005268200.jpg                    HUAT           15.00  15/06/2017   
3  X51005268262.jpg  HOME MASTER HARDWARE &           50.00  22/12/2017   
4  X51005268400.jpg  RESTORAN  HASSANBISTRO           15.00  12/28/2017   
5  X51005268472.jpg               ASIA MART           75.25   Not Found   
6  X51005301659.jpg  HOME MASTER HARDWARE &           50.00  22/12/2017   
7  X51005301661.jpg  LIGHTROOM  GALLERY SDN            4.13  20/12/2017   
8  X51005301667.jpg       LIGHTROOM GALLERY            2.25  20/11/2017   
9  X51005303661.jpg  LIGHTROOM  GALLERY SDN            4.13  20/12/2017   

          Category Ageing (Days)  As Of Date  
0           Others          2645  10-04-2026  
1           Others          3024  10-04-2026  
2           Others          3221 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Excel backup downloaded!
